In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('final.csv', parse_dates=True)

In [4]:
!pip install tsfresh

^C


In [12]:
import numpy as np
import pandas as pd
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import make_forecasting_frame
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')  # Вы можете выбрать другую стратегию заполнения, такую как 'median' или 'most_frequent'

In [7]:
# 1. Подготовка данных
df_shift, y = make_forecasting_frame(data["value"], kind="power", max_timeshift=10, rolling_direction=1)

# 2. Извлечение признаков
X = extract_features(df_shift, column_id="id", column_sort="time", column_value="value", impute_function=np.nanmean, show_warnings=False)


Feature Extraction: 100%|██████████| 15/15 [00:21<00:00,  1.44s/it]


In [8]:
X.head()

value__variance_larger_than_standard_deviation  \
id 1                                             0.0   
   2                                             1.0   
   3                                             1.0   
   4                                             1.0   
   5                                             1.0   

      value__has_duplicate_max  value__has_duplicate_min  \
id 1                       0.0                       0.0   
   2                       0.0                       0.0   
   3                       0.0                       0.0   
   4                       0.0                       0.0   
   5                       0.0                       0.0   

      value__has_duplicate  value__sum_values  value__abs_energy  \
id 1                   0.0         123.430000       15234.964900   
   2                   0.0         249.269996       31070.669493   
   3                   0.0         374.509994       46755.726592   
   4                   0.0         494.489997       61150.927712   
   5                   0.0         610.869994       74695.231414   

      value__mean_abs_change  value__mean_change  \
id 1                     NaN                 NaN   
   2                2.409996            2.409996   
   3                1.504997            0.904999   
   4                2.756663           -1.149999   
   5                2.967499           -1.762501   

      value__mean_second_derivative_central  value__median  ...  \
id 1                                    NaN     123.430000  ...   
   2                                    NaN     124.634998  ...   
   3                              -1.504997     125.239998  ...   
   4                              -1.917498     124.334999  ...   
   5                              -1.001667     123.430000  ...   

      value__fourier_entropy__bins_5  value__fourier_entropy__bins_10  \
id 1                             NaN                              NaN   
   2                       -0.000000                        -0.000000   
   3                        0.693147                         0.693147   
   4                        1.098612                         1.098612   
   5                        1.098612                         1.098612   

      value__fourier_entropy__bins_100  \
id 1                               NaN   
   2                         -0.000000   
   3                          0.693147   
   4                          1.098612   
   5                          1.098612   

      value__permutation_entropy__dimension_3__tau_1  \
id 1                                             NaN   
   2                                             NaN   
   3                                       -0.000000   
   4                                        0.693147   
   5                                        0.636514   

      value__permutation_entropy__dimension_4__tau_1  \
id 1                                             NaN   
   2                                             NaN   
   3                                             NaN   
   4                                       -0.000000   
   5                                        0.693147   

      value__permutation_entropy__dimension_5__tau_1  \
id 1                                             NaN   
   2                                             NaN   
   3                                             NaN   
   4                                             NaN   
   5                                            -0.0   

      value__permutation_entropy__dimension_6__tau_1  \
id 1                                             NaN   
   2                                             NaN   
   3                                             NaN   
   4                                             NaN   
   5                                             NaN   

      value__permutation_entropy__dimension_7__tau_1  \
id 1                                             NaN   
   2                           

In [9]:
df_shift.head()

,id,time,value,kind
1,"(id, 1)",0,123.430000,power
3,"(id, 2)",0,123.430000,power
4,"(id, 2)",1,125.839996,power
6,"(id, 3)",0,123.430000,power
7,"(id, 3)",1,125.839996,power


In [11]:
# 3. Разделение данных на обучающую и тестовую выборку
X.dropna(axis='columns', inplace=True)
split_index = int(0.8 * len(y))
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y[:split_index], y[split_index:]
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

In [13]:
# 4. Моделирование
regressor = RandomForestRegressor()
regressor.fit(X_train_imputed, y_train)
y_pred = regressor.predict(X_test_imputed)

# 5. Оценка модели
mse = mean_absolute_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

Mean Squared Error: 1.0377587350344881


In [19]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_absolute_error

# Проверьте размеры y_test и y_pred
print(f"Length of y_test: {len(y_test)}")
print(f"Length of y_pred: {len(y_pred)}")

# Убедитесь, что длины совпадают
if len(y_test) != len(y_pred):
    raise ValueError("y_test and y_pred must be of the same length.")

# Индексы для оси x
indices = np.arange(len(y_test))

plt.figure(figsize=(12, 6))

# Визуализируем реальные значения
plt.scatter(indices, y_test, color='blue', label='Real values', alpha=0.6)

# Визуализируем предсказанные значения
plt.scatter(indices, y_pred, color='red', label='Predicted values', alpha=0.6)

# Добавляем заголовки и легенду
plt.title('Real vs Predicted Values')
plt.xlabel('Index')
plt.ylabel('Value')
plt.legend()

# Отображаем график
plt.show()


Length of y_test: 0
Length of y_pred: 145


ValueError: y_test and y_pred must be of the same length.

AttributeError: 'DataFrame' object has no attribute 'iplot'